# MNIST and CNN

## MNIST

### Load Dataset

In [ ]:
import torchvision.datasets as dsets
import torchvision.transforms as transforms

In [ ]:
train_data = dsets.MNIST(root='data/',
                         train=True,
                         transform=transforms.ToTensor(),
                         download=True)

test_data = dsets.MNIST(root='data/',
                        train=False,
                        transform=transforms.ToTensor(),
                        download=True)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
fig = plt.figure(figsize = (15, 5))
ax1 = fig.add_subplot(1, 3, 1)
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)

ax1.set_title(train_data.targets[0].item())
ax1.imshow(train_data.data[0,:,:].numpy(), cmap='gray')

ax2.set_title(train_data.targets[1].item())
ax2.imshow(train_data.data[1,:,:].numpy(), cmap='gray')

ax3.set_title(train_data.targets[2].item())
ax3.imshow(train_data.data[2,:,:].numpy(), cmap='gray')

### Make Batches

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
batch_size = 100

train_loader = DataLoader(dataset=train_data,
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(dataset=test_data,
                         batch_size=batch_size,
                         shuffle=False)

In [ ]:
batch_images, batch_labels = iter(train_loader).next()
print(batch_labels.numpy(), ", ", len(batch_labels.numpy()))

## Train & Evaluate MLP

### Train MLP

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Linear(512, 1000),
    nn.ReLU(),
    nn.Linear(1000, 10)
)

In [ ]:
loss = nn.CrossEntropyLoss()
# def cross_entropy(input, target, weight=None, size_average=True, ignore_index=-100, reduce=True):

# Args:
#     input: Variable :math:`(N, C)` where `C = number of classes`
#     target: Variable :math:`(N)` where each value is
#         `0 <= targets[i] <= C-1`
#     weight (Tensor, optional): a manual rescaling weight given to each

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 5

In [ ]:
for epoch in range(num_epochs):
    
    total_batch = len(train_data) // batch_size
    
    for i, (batch_images, batch_labels) in enumerate(train_loader):
        
        x = batch_images.view(-1, 28 * 28)
        y = batch_labels
        
        pre = model(x)
        cost = loss(pre, y)
        
        optimizer.zero_grad()
        cost.backward()
        optimizer.step()
        
        if (i+1) % 300 == 0:
            print('Epoch [%d/%d], lter [%d/%d], Loss: %.4f'
                 %(epoch+1, num_epochs, i+1, total_batch, cost.item()))
    
print("Learning Finished!")

### Evaluate MLP

In [ ]:
correct = 0
total = 0

for images, labels in test_data:
    
    images  = images.view(-1, 28 * 28)
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += 1
    correct += (predicted == labels).sum()
    
print('Accuracy of test images: %f %%' % (100 * float(correct) / total))

In [ ]:
import random
r = random.randint(0, len(test_data)-1)
x_single = test_data.data[r:r + 1].view(-1,28*28).float()
y_single = test_data.targets[r:r + 1]

In [ ]:
pre_single = model(x_single)
plt.imshow(x_single.data.view(28,28).numpy(), cmap='gray')

print('Label : ', y_single.data.view(1).numpy())
print('Prediction : ', torch.max(pre_single.data, 1)[1].numpy())

## Train & Evaluate CNN

### Train CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layer = nn.Sequential(
            nn.Conv2d(1, 16, 5), # 1x28x28 -> 16x24x24
            nn.ReLU(),
            nn.Conv2d(16, 32, 5), # 16x24x24 -> 32x20x20
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 32x20x20 -> 
            nn.Conv2d(32, 64, 5),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.fc_layer = nn.Sequential(
            nn.Linear(64*3*3, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )       
        
    def forward(self,x):
        out = self.conv_layer(x)
        out = out.view(-1,64*3*3)
        out = self.fc_layer(out)

        return out

In [ ]:
model = CNN().cuda()

In [ ]:
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
num_epochs = 3

In [ ]:
for epoch in range(num_epochs):

    total_batch = len(train_data) // batch_size

    for i, (batch_images, batch_labels) in enumerate(train_loader):

        x = batch_images.cuda()
        y = batch_labels.cuda()

        pre = model(x)
        cost = loss(pre, y)

        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

        if (i+1) % 300 == 0:
            print('Epoch [%d/%d], lter [%d/%d], Loss: %.4f'
                 %(epoch+1, num_epochs, i+1, total_batch, cost.item()))

### Evaulate CNN

In [ ]:
correct = 0
total = 0

for images, labels in test_loader:
    
    images = images.cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Accuracy of test images: %f %%' % (100 * float(correct) / total))

## Save and Load Model

### Save Model

In [ ]:
torch.save(model, "sample1.pth")

In [ ]:
torch.save(model.state_dict(), "sample2.pth")

### Load Model

In [ ]:
torch.load("sample1.pth")

In [ ]:
# 클래스가 변경/제거되면 무조건 에러 (전체 객체 저장 방식)
# Always errors if class changes/disappears (whole-object save)
del CNN
torch.load("sample1.pth")

In [ ]:
# 클래스가 변경/제거되어도 불러올 수 있음 (state_dict 방식)
# Can be loaded even if the class changes/disappears (state_dict approach)
torch.load("sample2.pth")

In [ ]:
# 물론 최종적으로는 클래스 정의가 필요함
# Of course, the class definition is still required eventually
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, 5),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 5),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(64*5*5, 100),
            nn.ReLU(),
            nn.Linear(100, 10)              
        )
        
    def forward(self, x):
        out = self.conv_layer(x)
        out = out.view(-1, 64*5*5)
        out = self.fc_layer(out)
        
        return out
    
model = CNN().cuda()
model.load_state_dict(torch.load("sample2.pth"))

In [ ]:
model

---

# Appendix: `nn.Conv` Backpropagation 실습 (학부 26SS 중간고사 #8~#12)
# 부록: nn.Conv 역전파 실습 (학부 26SS 중간고사 #8~#12)

문제: $3 \times 3$ 입력에 대해 $2\times 2$ Conv (가중치=모두 1, bias 없음) → $2\times 2$ MaxPool → MSE 로 한 스텝 학습 (lr=0.5).
다음을 차례로 구하라:
- (#9) `pre.item()`
- (#10) `cost.item()`
- (#11) `model[0].weight.grad.sum().item()`  *(after `cost.backward()`)*
- (#12) `model[0].weight.data.sum().item()`  *(after `optimizer.step()`)*

Problem: train one step of a tiny CNN ($2\times 2$ Conv with all-ones weight, no bias → $2\times 2$ MaxPool → MSE, lr=0.5) on a single $3\times 3$ input. Find each of the values above.


## 1. 코드 (Code)

In [ ]:
# 학부 26SS 중간고사 #8 코드
# Undergrad 26SS midterm #8 code
import torch
import torch.nn as nn

x = torch.tensor([[[[1., 0., 2.],
                    [0., 3., 0.],
                    [2., 0., 1.]]]])
y = torch.tensor([[[[4.]]]])

model = nn.Sequential(
    nn.Conv2d(1, 1, 2, bias=False),   # 2x2 conv, no bias
    nn.MaxPool2d(2, 2),
)
# 필터를 모두 1로 강제 / force the filter to be all ones
model[0].weight.data = torch.ones_like(model[0].weight.data)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
loss = nn.MSELoss()      # 손실함수 / loss function

pre  = model(x)
cost = loss(pre, y)

optimizer.zero_grad()
cost.backward()
optimizer.step()


## 2. Forward 단계별 풀이 (Step-by-step forward)

### 2-1. Conv 계산 — 입력의 4개 $2\times 2$ 패치에 가중치(1로 채움)와 곱한 합

Each output cell of the conv = sum of an input $2\times 2$ patch (since all weights are 1).

$$
x = \begin{pmatrix} 1 & 0 & 2 \\ 0 & 3 & 0 \\ 2 & 0 & 1 \end{pmatrix}, \quad
W = \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}
$$

| 출력 위치 (row, col) | 입력 패치 | 합 (=conv 값) |
|---|---|---|
| (0,0) | $\bigl(\begin{smallmatrix}1&0\\0&3\end{smallmatrix}\bigr)$ | **4** |
| (0,1) | $\bigl(\begin{smallmatrix}0&2\\3&0\end{smallmatrix}\bigr)$ | **5** |
| (1,0) | $\bigl(\begin{smallmatrix}0&3\\2&0\end{smallmatrix}\bigr)$ | **5** |
| (1,1) | $\bigl(\begin{smallmatrix}3&0\\0&1\end{smallmatrix}\bigr)$ | **4** |

따라서 conv 출력은 $\bigl(\begin{smallmatrix}4&5\\5&4\end{smallmatrix}\bigr)$.

### 2-2. MaxPool 2×2 → 최댓값 1개 출력
The $2\times 2$ MaxPool over the $2\times 2$ conv output simply returns its max.
$$\max\{4,5,5,4\} = 5 \;\Rightarrow\; \textbf{pre.item() = 5}.$$

### 2-3. MSE
$$\text{cost} = (5 - 4)^2 = 1 \;\Rightarrow\; \textbf{cost.item() = 1}.$$


## 3. Backward 단계별 풀이 (Step-by-step backward)

체인 룰을 위에서 아래로 따라간다.
We chase the chain rule from top to bottom.

### 3-1. MSE 의 미분
$\dfrac{\partial \text{cost}}{\partial \text{pre}} = 2(\text{pre} - y) = 2(5 - 4) = 2.$

(주의: PyTorch `MSELoss` 의 기본 reduction='mean'. 출력이 스칼라(원소 1개)라 mean 과 sum 이 동일.)
(Note: PyTorch's `MSELoss` uses `reduction='mean'` by default. Since the output has only one element, mean and sum coincide here.)

### 3-2. MaxPool 의 backward — argmax 위치로만 라우팅
MaxPool routes the upstream gradient **only to the argmax** location; every other position gets 0.

5 가 두 곳(0,1) 과 (1,0) 에 동시에 있는 **타이(tie)** 상황. PyTorch 의 MaxPool 은 같은 값일 때 **row-major 순서로 가장 먼저 나오는 위치**를 선택한다 → 위치 **(0,1)** 이 선택됨.

When there is a tie, PyTorch's MaxPool picks the **first** maximum in row-major order, which is position **(0,1)**.

따라서 conv 출력에 대한 기울기:
$$\frac{\partial \text{cost}}{\partial \text{conv}} =
\begin{pmatrix} 0 & 2 \\ 0 & 0 \end{pmatrix}.$$

### 3-3. Conv 의 backward — 각 출력 위치의 입력 패치를 가중평균

선형층과 동일하게 $\partial(W \ast X)/\partial W$ 는 입력 패치이다.
Just as in the linear case, $\partial(W \ast X)/\partial W$ at output position $(i,j)$ equals the input patch at $(i,j)$.

$$
\frac{\partial \text{cost}}{\partial W} = \sum_{(i,j)} \frac{\partial \text{cost}}{\partial \text{conv}_{ij}} \cdot \text{patch}(i,j).
$$

비-0 항은 (0,1) 하나뿐이므로:
Only the (0,1) term is non-zero, so:

$$
\nabla_W = 2 \cdot \begin{pmatrix} 0 & 2 \\ 3 & 0 \end{pmatrix}
       = \begin{pmatrix} 0 & 4 \\ 6 & 0 \end{pmatrix}.
$$

$$\textbf{weight.grad.sum() = 0+4+6+0 = 10}.$$

### 3-4. SGD 업데이트
$W \leftarrow W - \eta \nabla_W$ 와 $\eta = 0.5$:
$$
W_{\text{new}}
= \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}
- 0.5 \cdot \begin{pmatrix} 0 & 4 \\ 6 & 0 \end{pmatrix}
= \begin{pmatrix} 1 & -1 \\ -2 & 1 \end{pmatrix}.
$$

$$\textbf{weight.data.sum() = 1 + (-1) + (-2) + 1 = -1}.$$


## 4. 코드로 검증 (Verify with code)

In [ ]:
# 위 풀이가 맞는지 한 번 더 출력해서 확인
# Print to verify the hand-calculated answers above
import torch
import torch.nn as nn

x = torch.tensor([[[[1., 0., 2.],
                    [0., 3., 0.],
                    [2., 0., 1.]]]])
y = torch.tensor([[[[4.]]]])

model = nn.Sequential(
    nn.Conv2d(1, 1, 2, bias=False),
    nn.MaxPool2d(2, 2),
)
model[0].weight.data = torch.ones_like(model[0].weight.data)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
loss = nn.MSELoss()

pre  = model(x)
cost = loss(pre, y)
print("pre.item()  =", pre.item())          # 5.0
print("cost.item() =", cost.item())         # 1.0

optimizer.zero_grad()
cost.backward()
print("weight.grad =\n", model[0].weight.grad)
print("weight.grad.sum() =", model[0].weight.grad.sum().item())   # 10.0

optimizer.step()
print("weight (after step) =\n", model[0].weight.data)
print("weight.sum() =", model[0].weight.data.sum().item())        # -1.0


---

## 5. 변형: MaxPool → AvgPool 일 때
## 5. Variant: replacing MaxPool with AvgPool

같은 입력·필터·target 을 사용하되 풀링만 `nn.AvgPool2d(2, 2)` 로 바꿔보자.
We keep input/filter/target the same and swap the pool to `nn.AvgPool2d(2, 2)`.

### 5-1. 무엇이 달라지는가? (What changes?)

- **MaxPool**: 출력 영역 안의 최댓값 1 개를 통과시키고, 역전파 시 기울기는 argmax 위치로만 라우팅.
- **AvgPool**: 출력 영역의 평균을 통과시키고, 역전파 시 기울기는 영역 내 모든 위치에 **균등 분배** ($1/k^2$ 배).

- **MaxPool**: forward picks the single max; backward routes the gradient **only to the argmax**.
- **AvgPool**: forward returns the mean; backward distributes the gradient **equally** across the pool window ($\times 1/k^2$).


### 5-2. Forward
$$\text{pre} = \frac{4 + 5 + 5 + 4}{4} = 4.5 \;\Rightarrow\; \textbf{pre.item() = 4.5}.$$
$$\text{cost} = (4.5 - 4)^2 = 0.25 \;\Rightarrow\; \textbf{cost.item() = 0.25}.$$

### 5-3. Backward
$$\frac{\partial \text{cost}}{\partial \text{pre}} = 2(4.5 - 4) = 1.0.$$

AvgPool backward → $1/4$ 씩 균등 분배:
$$\frac{\partial \text{cost}}{\partial \text{conv}}
= 1.0 \cdot \frac{1}{4}
\begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}
= \begin{pmatrix} 0.25 & 0.25 \\ 0.25 & 0.25 \end{pmatrix}.$$

이제 모든 출력 위치의 입력 패치를 그 기울기로 가중평균:
Now we multiply each input patch by the corresponding gradient and sum.

$$
\nabla_W
= 0.25 \cdot \Bigl[
\bigl(\begin{smallmatrix}1&0\\0&3\end{smallmatrix}\bigr)
+ \bigl(\begin{smallmatrix}0&2\\3&0\end{smallmatrix}\bigr)
+ \bigl(\begin{smallmatrix}0&3\\2&0\end{smallmatrix}\bigr)
+ \bigl(\begin{smallmatrix}3&0\\0&1\end{smallmatrix}\bigr) \Bigr].
$$

(편의 사실: 입력 패치 4 개의 합은 conv 출력 자체와 동일하다 — 즉 $\bigl(\begin{smallmatrix}4&5\\5&4\end{smallmatrix}\bigr)$.)
(Useful fact: the sum of the four input patches equals the conv output itself.)

$$
\nabla_W
= 0.25 \cdot \begin{pmatrix} 4 & 5 \\ 5 & 4 \end{pmatrix}
= \begin{pmatrix} 1.00 & 1.25 \\ 1.25 & 1.00 \end{pmatrix}.
$$

$$\textbf{weight.grad.sum() = 1.00 + 1.25 + 1.25 + 1.00 = 4.5}.$$

### 5-4. SGD 업데이트 (lr=0.5)
$$
W_{\text{new}}
= \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}
- 0.5 \cdot \begin{pmatrix} 1.00 & 1.25 \\ 1.25 & 1.00 \end{pmatrix}
= \begin{pmatrix} 0.500 & 0.375 \\ 0.375 & 0.500 \end{pmatrix}.
$$
$$\textbf{weight.data.sum() = 0.500 + 0.375 + 0.375 + 0.500 = 1.75}.$$


### 5-5. 코드로 검증 (Verify with code)

In [ ]:
# AvgPool 변형 — 풀링만 nn.AvgPool2d 로 교체
# AvgPool variant — only the pool is replaced with nn.AvgPool2d
import torch
import torch.nn as nn

x = torch.tensor([[[[1., 0., 2.],
                    [0., 3., 0.],
                    [2., 0., 1.]]]])
y = torch.tensor([[[[4.]]]])

model = nn.Sequential(
    nn.Conv2d(1, 1, 2, bias=False),
    nn.AvgPool2d(2, 2),
)
model[0].weight.data = torch.ones_like(model[0].weight.data)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
loss = nn.MSELoss()

pre  = model(x)
cost = loss(pre, y)
print("pre.item()  =", pre.item())          # 4.5
print("cost.item() =", cost.item())         # 0.25

optimizer.zero_grad()
cost.backward()
print("weight.grad =\n", model[0].weight.grad)
print("weight.grad.sum() =", model[0].weight.grad.sum().item())   # 4.5

optimizer.step()
print("weight (after step) =\n", model[0].weight.data)
print("weight.sum() =", model[0].weight.data.sum().item())        # 1.75


## 6. 핵심 정리 (Takeaway)

| 항목 | MaxPool | AvgPool |
|---|---|---|
| forward | 최댓값 1 개 | 평균 |
| backward 경로 | argmax 위치 1 곳만 | 모든 위치 균등 ($1/k^2$) |
| 입력에 변화가 거의 없는 위치 | 무시됨 (gradient 0) | 영향 줌 (gradient 0 아님) |
| 학습 효과 | 가장 강한 신호에 집중 | 영역 전체에서 부드럽게 평균 |

**실무 한 줄 요약**: 같은 conv 출력이라도 풀링 종류에 따라 가중치가 학습되는 방식이 완전히 달라진다. 시험에서는 forward 결과뿐 아니라 **어디로 기울기가 흐르는가**를 정확히 추적해야 한다.

**One-line takeaway**: even with the same conv output, the choice of pooling completely changes how the weight learns. In an exam you must trace **where the gradient flows**, not just the forward value.
